# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [1]:
# importar librerías
import pandas as pd
import numpy as np
from IPython.display import display

In [2]:
# cargar archivos
orders = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv") # tu código aquí
catalog = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv") # tu código aquí
marketing = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv") # tu código aquí

In [3]:
# explorar datasets
display(orders.head())
orders.info() # tu código aquí

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


In [4]:
display(catalog.head())
catalog.info() 

,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


In [5]:
display(marketing.head())
marketing.info() 

,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

In [6]:
print("DUPLICADOS:")
print(orders.duplicated().sum())

print("\nVALORES NULOS:")
print(orders.isnull().sum())

print("\nTIPOS DE DATOS:")
print(orders.dtypes)

print("\nESTADÍSTICAS NUMÉRICAS:")
display(orders.describe())  # tu código aquí

DUPLICADOS:
100

VALORES NULOS:
id_pedido               0
id_usuario              0
fecha_hora_pedido       0
pais                  300
dispositivo            20
fuente_referencia      30
nombre_producto        30
categoria_producto     80
cantidad               50
precio_unitario        50
monto_descuento        50
monto_total             0
dtype: int64

TIPOS DE DATOS:
id_pedido              object
id_usuario             object
fecha_hora_pedido      object
pais                   object
dispositivo            object
fuente_referencia      object
nombre_producto        object
categoria_producto     object
cantidad              float64
precio_unitario       float64
monto_descuento       float64
monto_total           float64
dtype: object

ESTADÍSTICAS NUMÉRICAS:


,cantidad,precio_unitario,monto_descuento,monto_total
count,25050.000000,25050.000000,25050.000000,2.510000e+04
mean,7.092735,259.305549,4.500798,2.072680e+03
std,296.277003,138.726461,5.223010,9.894995e+04
min,-2.000000,20.030000,0.000000,-4.926500e+02
25%,1.000000,138.377500,0.000000,1.805075e+02
50%,2.000000,258.715000,0.000000,3.417500e+02
75%,2.000000,380.332500,10.000000,5.185800e+02
max,20000.000000,499.960000,15.000000,8.840200e+06


In [7]:
print("Filas con cantidad negativa:", (orders['cantidad'] < 0).sum())
print("Filas con monto_total negativo:", (orders['monto_total'] < 0).sum())

Filas con cantidad negativa: 4
Filas con monto_total negativo: 4


In [8]:
orders[(orders['cantidad'] < 0) | (orders['monto_total'] < 0)]

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
266,order_266,user_7011,2025-03-13,NaN,desktop,paid_search,Phone-Pro-128GB,Electronica,-2.0,101.31,10.0,-192.62
267,order_267,user_1087,2025-05-07,NaN,desktop,social,Phone-Pro-128GB,Electronica,-1.0,43.50,5.0,-38.50
268,order_268,user_84,2025-02-19,NaN,desktop,organic,Phone-Pro-128GB,Electronica,-1.0,497.65,5.0,-492.65
269,order_269,user_3323,2025-05-25,NaN,desktop,paid_search,Phone-Pro-128GB,Electronica,-1.0,423.53,0.0,-423.53


In [9]:
orders.loc[
    (orders['cantidad'] < 0) | (orders['monto_total'] < 0),
    ['cantidad', 'precio_unitario', 'monto_total']
]

,cantidad,precio_unitario,monto_total
266,-2.0,101.31,-192.62
267,-1.0,43.50,-38.50
268,-1.0,497.65,-492.65
269,-1.0,423.53,-423.53


In [10]:
orders.loc[
    (orders['cantidad'] < 0) | (orders['monto_total'] < 0),
    ['cantidad', 'precio_unitario', 'monto_descuento', 'monto_total']
]

,cantidad,precio_unitario,monto_descuento,monto_total
266,-2.0,101.31,10.0,-192.62
267,-1.0,43.50,5.0,-38.50
268,-1.0,497.65,5.0,-492.65
269,-1.0,423.53,0.0,-423.53


In [11]:
filas_negativas = orders['cantidad'] < 0

orders.loc[filas_negativas, 'cantidad'] = (
    orders.loc[filas_negativas, 'cantidad'].abs()
)

orders.loc[filas_negativas, 'monto_total'] = (
    orders.loc[filas_negativas, 'cantidad'] *
    orders.loc[filas_negativas, 'precio_unitario'] -
    orders.loc[filas_negativas, 'monto_descuento']
)

In [12]:
print("Cantidad negativas:", (orders['cantidad'] < 0).sum())
print("Monto total negativo:", (orders['monto_total'] < 0).sum())

Cantidad negativas: 0
Monto total negativo: 0


In [13]:
print(orders['fecha_hora_pedido'].dtype)

object


In [14]:
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'])

In [15]:
orders['fecha_hora_pedido'].head()

0   2025-05-22
1   2025-06-15
2   2025-05-02
3   2025-06-09
4   2025-03-30
Name: fecha_hora_pedido, dtype: datetime64[ns]

In [16]:
orders['fecha_hora_pedido'].isnull().sum()

0

In [17]:
orders[orders.duplicated()]

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
25000,order_22936,user_4028,2025-06-30,Argentina,desktop,social,Vacuum-Pro-Black,Hogar,2.0,41.91,10.0,73.82
25001,order_13710,user_4466,2025-03-20,Mexico,desktop,social,Jacket-Winter-M,Moda,2.0,483.32,0.0,966.65
25002,order_14562,user_6590,2025-03-23,Mexico,mobile,paid_search,Tablet-Standard-64GB,Electronica,1.0,481.96,5.0,476.96
25003,order_11537,user_3115,2025-03-08,Argentina,desktop,organic,Vacuum-Pro-Black,Hogar,1.0,147.20,5.0,142.20
25004,order_4533,user_7944,2025-02-16,Argentina,mobile,organic,Blender-XL-Red,Hogar,1.0,176.91,5.0,171.91
...,...,...,...,...,...,...,...,...,...,...,...,...
25095,order_3913,user_380,2025-02-18,Argentina,desktop,paid_search,Phone-Pro-128GB,Electronica,1.0,82.28,0.0,82.28
25096,order_23405,user_7833,2025-04-04,Colombia,mobile,paid_search,Phone-Pro-128GB,Electronica,2.0,99.25,5.0,193.50
25097,order_5615,user_5417,2025-05-13,Colombia,desktop,social,Blender-XL-Red,Hogar,2.0,450.35,5.0,895.69
25098,order_812,user_1530,2025-03-31,Argentina,desktop,organic,Tablet-Standard-64GB,Electronica,1.0,167.32,10.0,157.32


In [18]:
orders = orders.drop_duplicates()

In [19]:
orders.duplicated().sum()

0

In [20]:
orders[orders['pais'].isnull()].head(10)

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
124,order_124,user_6671,2025-04-12,NaN,mobile,organic,Blender-XL-Red,Hogar,1.0,298.44,5.0,293.44
125,order_125,user_5263,2025-05-30,NaN,mobile,organic,Tablet-Standard-64GB,Electronica,1.0,133.13,0.0,133.13
126,order_126,user_4952,2025-04-19,NaN,desktop,paid_search,Phone-Pro-128GB,Electronica,2.0,177.23,0.0,354.45
127,order_127,user_2619,2025-06-26,NaN,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,329.53,10.0,319.53
128,order_128,user_2444,2025-05-06,NaN,desktop,social,Vacuum-Pro-Black,Hogar,2.0,218.43,0.0,436.86
129,order_129,user_7157,2025-04-17,NaN,desktop,paid_search,Sneakers-Urban-42,Moda,1.0,367.88,10.0,357.88
130,order_130,user_3279,2025-02-06,NaN,desktop,paid_search,Vacuum-Pro-Black,Hogar,1.0,114.83,0.0,114.83
131,order_131,user_2914,2025-05-05,NaN,mobile,organic,Blender-XL-Red,Hogar,1.0,117.84,10.0,107.84
132,order_132,user_2173,2025-02-03,NaN,desktop,organic,Tablet-Standard-64GB,Electronica,1.0,383.84,0.0,383.84
133,order_133,user_4038,2025-05-12,NaN,mobile,social,Vacuum-Pro-Black,Hogar,2.0,345.85,0.0,691.69


In [21]:
usuarios_sin_pais = orders.loc[orders['pais'].isnull(), 'id_usuario'].unique()

orders[
    orders['id_usuario'].isin(usuarios_sin_pais)
].groupby('id_usuario')['pais'].apply(lambda x: x.dropna().unique())

id_usuario
user_1010          [Mexico, mexico]
user_1032    [argentina, Argentina]
user_1065    [Argentina, argentina]
user_1087                  [Mexico]
user_113           [Mexico, mexico]
                      ...          
user_90            [Mexico, mexico]
user_907     [argentina, Argentina]
user_939                   [Mexico]
user_967                [Argentina]
user_999                [Argentina]
Name: pais, Length: 291, dtype: object

In [22]:
orders['pais'] = orders['pais'].str.strip().str.lower()

In [23]:
orders['pais'].value_counts(dropna=False)

mexico       8341
colombia     8304
argentina    8055
NaN           300
Name: pais, dtype: int64

In [24]:
orders['pais'] = orders['pais'].fillna(
    orders.groupby('id_usuario')['pais'].transform('first')
)

In [25]:
orders['pais'] = orders['pais'].fillna('desconocido')

In [26]:
orders['pais'].isnull().sum()

0

In [27]:
orders['pais'].value_counts()

mexico         8440
colombia       8386
argentina      8155
desconocido      19
Name: pais, dtype: int64

In [28]:
print("DUPLICADOS:")
print(catalog.duplicated().sum())

print("\nVALORES NULOS:")
print(catalog.isnull().sum())

print("\nTIPOS DE DATOS:")
print(catalog.dtypes)

print("\nESTADÍSTICAS NUMÉRICAS:")
display(catalog.describe())

DUPLICADOS:
0

VALORES NULOS:
nombre_producto       0
categoria_producto    0
costo_unitario        0
proveedor             0
dtype: int64

TIPOS DE DATOS:
nombre_producto        object
categoria_producto     object
costo_unitario        float64
proveedor              object
dtype: object

ESTADÍSTICAS NUMÉRICAS:


,costo_unitario
count,7.000000
mean,102.252857
std,111.011563
min,10.120000
25%,16.905000
50%,25.210000
75%,182.975000
max,280.680000


In [29]:

print("DUPLICADOS:")
print(marketing.duplicated().sum())

print("\nVALORES NULOS:")
print(marketing.isnull().sum())

print("\nTIPOS DE DATOS:")
print(marketing.dtypes)

print("\nESTADÍSTICAS NUMÉRICAS:")
display(marketing.describe())

DUPLICADOS:
0

VALORES NULOS:
fecha           0
pais            0
id_campaña      0
canal         101
gasto           0
dtype: int64

TIPOS DE DATOS:
fecha          object
pais           object
id_campaña     object
canal          object
gasto         float64
dtype: object

ESTADÍSTICAS NUMÉRICAS:


,gasto
count,1620.00000
mean,1772.74292
std,734.43294
min,501.11000
25%,1128.03000
50%,1782.42500
75%,2420.68500
max,2999.36000


In [30]:
marketing['canal'].value_counts(dropna=False)

paid_search    507
organic        506
social         506
NaN            101
Name: canal, dtype: int64

In [31]:
marketing[marketing['canal'].isnull()].head(10) 

,fecha,pais,id_campaña,canal,gasto
98,2025-01-11,Argentina,social_Argentina,NaN,849.70
99,2025-01-12,Mexico,organic_Mexico,NaN,2033.56
100,2025-01-12,Mexico,paid_search_Mexico,NaN,1260.65
101,2025-01-12,Mexico,social_Mexico,NaN,1660.90
102,2025-01-12,Colombia,organic_Colombia,NaN,1819.27
103,2025-01-12,Colombia,paid_search_Colombia,NaN,1583.33
104,2025-01-12,Colombia,social_Colombia,NaN,712.64
105,2025-01-12,Argentina,organic_Argentina,NaN,1860.40
106,2025-01-12,Argentina,paid_search_Argentina,NaN,885.06
107,2025-01-12,Argentina,social_Argentina,NaN,2253.77


In [32]:
marketing.loc[marketing['canal'].isnull(), 'canal'] = (
    marketing.loc[marketing['canal'].isnull(), 'id_campaña']
    .str.rsplit('_', n=1).str[0]
)

In [33]:
marketing['canal'].isnull().sum()

0

In [34]:
marketing['canal'].value_counts()

organic        540
paid_search    540
social         540
Name: canal, dtype: int64

---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [35]:
orders[orders['nombre_producto'] == '30']

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total


In [36]:
orders[orders['categoria_producto'].isnull()][['nombre_producto', 'categoria_producto']]

,nombre_producto,categoria_producto
44,NaN,NaN
45,NaN,NaN
46,NaN,NaN
47,NaN,NaN
48,NaN,NaN
...,...,...
119,Sneakers-Urban-42,NaN
120,Sneakers-Urban-42,NaN
121,Sneakers-Urban-42,NaN
122,Sneakers-Urban-42,NaN


In [37]:
orders['nombre_producto'] = orders['nombre_producto'].fillna('Desconocido')
orders['categoria_producto'] = orders['categoria_producto'].fillna('Desconocido')
orders['fuente_referencia'] = orders['fuente_referencia'].fillna('Desconocido')
orders['dispositivo'] = orders['dispositivo'].fillna('Desconocido')

In [38]:
orders[['nombre_producto',
        'categoria_producto',
        'fuente_referencia',
        'dispositivo']].isnull().sum()

nombre_producto       0
categoria_producto    0
fuente_referencia     0
dispositivo           0
dtype: int64

In [39]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

In [40]:
import pandas as pd
print(pd.read_csv('marketing_clean.csv')['canal'].isnull().sum())
print(pd.read_csv('orders_clean.csv')['pais'].value_counts())

0
mexico         8440
colombia       8386
argentina      8155
desconocido      19
Name: pais, dtype: int64


---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [41]:
orders_catalog = orders.merge(
    catalog[['nombre_producto', 'costo_unitario']],
    on='nombre_producto',
    how='left'
) # tu código aquí


In [42]:
orders_catalog[['nombre_producto', 'cantidad', 'monto_total', 'costo_unitario']].head()

,nombre_producto,cantidad,monto_total,costo_unitario
0,Jacket-Winter-M,2.0,665.37,189.31
1,Tablet-Standard-64GB,1.0,171.86,25.21
2,Blender-XL-Red,2.0,195.99,176.64
3,Tablet-Standard-64GB,1.0,242.87,25.21
4,Blender-XL-Red,1.0,336.28,176.64


In [43]:
orders.shape

(25000, 12)

In [44]:
orders_catalog.shape

(25000, 13)

In [45]:
orders_catalog['costo_unitario'].isnull().sum()

30

In [46]:
orders_catalog[orders_catalog['costo_unitario'].isnull()][
    ['nombre_producto', 'cantidad', 'monto_total', 'costo_unitario']
]

,nombre_producto,cantidad,monto_total,costo_unitario
44,Desconocido,1.0,313.38,NaN
45,Desconocido,2.0,698.12,NaN
46,Desconocido,1.0,349.31,NaN
47,Desconocido,2.0,953.56,NaN
48,Desconocido,2.0,275.93,NaN
49,Desconocido,2.0,867.53,NaN
50,Desconocido,1.0,490.38,NaN
51,Desconocido,2.0,765.89,NaN
52,Desconocido,2.0,830.01,NaN
53,Desconocido,2.0,472.98,NaN


In [47]:
filas_nulas = orders_catalog[orders_catalog['nombre_producto'].isna()]

monto_nulos = filas_nulas['monto_total'].sum()
monto_total = orders_catalog['monto_total'].sum()

porcentaje_nulos = (monto_nulos / monto_total) * 100

print("Monto de las filas con producto nulo:", monto_nulos)
print("Monto total:", monto_total)
print("Porcentaje que representan:", porcentaje_nulos)

Monto de las filas con producto nulo: 0.0
Monto total: 51989756.859999985
Porcentaje que representan: 0.0


In [48]:
orders_rentabilidad = orders_catalog.dropna(subset=['nombre_producto']).copy()

orders_rentabilidad

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,costo_unitario
0,order_0,user_6993,2025-05-22,argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37,189.31
1,order_1,user_1329,2025-06-15,mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86,25.21
2,order_2,user_3194,2025-05-02,argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99,176.64
3,order_3,user_4510,2025-06-09,colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87,25.21
4,order_4,user_5044,2025-03-30,argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28,176.64
...,...,...,...,...,...,...,...,...,...,...,...,...,...
24995,order_24995,user_4217,2025-04-16,argentina,mobile,social,Tablet-Standard-64GB,Electronica,1.0,64.57,0.0,64.57,25.21
24996,order_24996,user_3945,2025-02-14,argentina,mobile,social,Vacuum-Pro-Black,Hogar,2.0,292.90,15.0,570.81,16.60
24997,order_24997,user_3119,2025-03-27,colombia,mobile,social,Jacket-Winter-M,Moda,1.0,226.85,0.0,226.85,189.31
24998,order_24998,user_7218,2025-02-01,mexico,mobile,social,Sneakers-Urban-42,Moda,2.0,418.06,0.0,836.12,17.21


In [49]:
orders_rentabilidad['costo_total'] = (
    orders_rentabilidad['cantidad'] * orders_rentabilidad['costo_unitario']
)

In [50]:
orders_rentabilidad['ganancia_bruta'] = (
    orders_rentabilidad['monto_total'] - orders_rentabilidad['costo_total']
)

In [51]:
orders_rentabilidad[orders_rentabilidad['nombre_producto'].isnull()][
    ['nombre_producto', 'costo_total', 'ganancia_bruta']
]

,nombre_producto,costo_total,ganancia_bruta


In [52]:
revenue_total = orders_rentabilidad['monto_total'].sum()
revenue_total

51989756.859999985

In [53]:
costo_total = orders_rentabilidad['costo_total'].sum()
costo_total

43124119.60999999

In [54]:
marketing_total = marketing['gasto'].sum()
marketing_total

2871843.53

In [55]:
profit_total = revenue_total - costo_total - marketing_total
profit_total

5993793.719999993

In [56]:
margen_profit = (profit_total / revenue_total) * 100
margen_profit

11.528797367028108

#  Rentabilidad del negocio
- Revenue total: $51,977,494.24
- Costo total: $43,124,119.61
- Inversión total en marketing: $2,871,843.53
- Profit total: $5,981,531.10
- Margen de profit: 11.53%

 Conclusión: 
RappiPlus presenta una rentabilidad positiva. Después de cubrir los costos de los productos y la inversión en marketing, el negocio obtiene un profit de aproximadamente $5.99 millones, equivalente a un margen de 11.53% sobre el revenue.

In [57]:
ticket_promedio = orders_rentabilidad['monto_total'].mean()
ticket_promedio

2079.5902743999995

In [58]:
cantidad_promedio = orders_rentabilidad['cantidad'].mean()
cantidad_promedio

7.115511022044088

# Comportamiento de Ventas
- Ticket promedio por orden: $2,081.60
- Cantidad promedio de productos por orden: 7.12 productos.

 Conclusión:
En promedio, cada orden genera aproximadamente $2,079.59 y contiene alrededor de 7 productos, lo que indica que los pedidos suelen incluir múltiples productos.

In [59]:
producto_mas_vendido = (
    orders_rentabilidad.groupby('nombre_producto')['cantidad']
    .sum()
    .sort_values(ascending=False)
)

producto_mas_vendido.head(10)

nombre_producto
Laptop-Gaming-16GB      144198.0
Vacuum-Pro-Black          6284.0
Blender-XL-Red            6279.0
Jacket-Winter-M           6256.0
Sneakers-Urban-42         6172.0
Tablet-Standard-64GB      4153.0
Phone-Pro-128GB           4145.0
Desconocido                 45.0
Name: cantidad, dtype: float64

In [60]:
laptop = orders_rentabilidad[
    orders_rentabilidad['nombre_producto'] == 'Laptop-Gaming-16GB'
]

print("Número de filas:", len(laptop))
print("Cantidad máxima:", laptop['cantidad'].max())
print("Estadísticas de cantidad:")
laptop['cantidad'].describe()

Número de filas: 2782
Cantidad máxima: 20000.0
Estadísticas de cantidad:


count     2778.000000
mean        51.907127
std        888.554469
min          1.000000
25%          1.000000
50%          2.000000
75%          2.000000
max      20000.000000
Name: cantidad, dtype: float64

In [61]:
orders_rentabilidad[
    (orders_rentabilidad['nombre_producto'] == 'Laptop-Gaming-16GB') &
    (orders_rentabilidad['cantidad'] > 100)
][
    ['nombre_producto', 'cantidad', 'precio_unitario', 'monto_total']
].sort_values('cantidad', ascending=False)

,nombre_producto,cantidad,precio_unitario,monto_total
3656,Laptop-Gaming-16GB,20000.0,297.66,5953200.0
3668,Laptop-Gaming-16GB,20000.0,348.31,6966200.0
3722,Laptop-Gaming-16GB,20000.0,442.01,8840200.0
3726,Laptop-Gaming-16GB,20000.0,290.85,5817000.0
3521,Laptop-Gaming-16GB,10000.0,43.14,431400.0
3522,Laptop-Gaming-16GB,10000.0,280.55,2805500.0
3586,Laptop-Gaming-16GB,10000.0,490.35,4903500.0
3643,Laptop-Gaming-16GB,10000.0,238.15,2381500.0
3689,Laptop-Gaming-16GB,10000.0,87.69,876900.0
3748,Laptop-Gaming-16GB,10000.0,336.93,3369300.0


In [62]:
ventas_sin_atipicos = (
    orders_rentabilidad[orders_rentabilidad['cantidad'] <= 100]
    .groupby('nombre_producto')['cantidad']
    .sum()
    .sort_values(ascending=False)
)

ventas_sin_atipicos.head(10)

nombre_producto
Vacuum-Pro-Black        6284.0
Blender-XL-Red          6279.0
Jacket-Winter-M         6256.0
Sneakers-Urban-42       6172.0
Laptop-Gaming-16GB      4198.0
Tablet-Standard-64GB    4153.0
Phone-Pro-128GB         4145.0
Desconocido               45.0
Name: cantidad, dtype: float64

# Producto más vendido

- El análisis inicial mostró que Laptop-Gaming-16GB acumulaba 144,198 unidades. Sin embargo, se identificaron 10 registros con cantidades extremadamente altas (4 órdenes de 20,000 unidades y 6 de 10,000 unidades), que representaban aproximadamente el 97.09% de las unidades reportadas para este producto.

- Para evitar que estos valores atípicos distorsionaran el análisis, se realizó una revisión   excluyendo temporalmente cantidades superiores a 100 unidades.

- Bajo este criterio, el producto con mayor volumen de ventas fue Vacuum-Pro-Black, con 6,284 unidades.

 Conclusión:
Vacuum-Pro-Black se considera el producto más vendido bajo el análisis depurado. Los registros atípicos de Laptop-Gaming-16GB se conservaron en los datos originales y se documentaron como una posible anomalía que requiere validación adicional.

In [63]:
gasto_por_canal = (
    marketing.groupby('canal')['gasto']
    .sum()
    .sort_values(ascending=False)
)

gasto_por_canal

canal
social         976818.37
organic        972650.96
paid_search    922374.20
Name: gasto, dtype: float64

In [64]:
marketing['gasto'].describe()

count    1620.00000
mean     1772.74292
std       734.43294
min       501.11000
25%      1128.03000
50%      1782.42500
75%      2420.68500
max      2999.36000
Name: gasto, dtype: float64

# Gasto de marketing por canal
- Social: $976,818.37
- Organic: $972,650.96
- Paid Search: $922,374.20

 Conclusión: 
La inversión en marketing se encuentra relativamente distribuida entre los tres canales. Social concentra el mayor gasto, seguido de Organic y Paid Search.

---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [65]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [66]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [67]:
query_eventos = '''
SELECT 
    nombre_evento,
    COUNT(*) AS cantidad_eventos
FROM events
GROUP BY nombre_evento
ORDER BY cantidad_eventos DESC;
'''

eventos = pd.read_sql(query_eventos, con=engine)
eventos

,nombre_evento,cantidad_eventos
0,first_visit,29957
1,add_to_cart,24157
2,select_item,23887
3,begin_checkout,17971
4,add_payment_info,12018
5,purchase,12010


In [68]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT 
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM events
GROUP BY nombre_evento
ORDER BY CASE nombre_evento
    WHEN 'first_visit' THEN 1
    WHEN 'add_to_cart' THEN 2
    WHEN 'select_item' THEN 3
    WHEN 'begin_checkout' THEN 4
    WHEN 'add_payment_info' THEN 5
    WHEN 'purchase' THEN 6
END;
'''                             
    
totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,usuarios_unicos
0,first_visit,7796
1,add_to_cart,7634
2,select_item,7582
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [69]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH funnel AS (
    SELECT 
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios_unicos
    FROM events
    GROUP BY nombre_evento
),
funnel_ordenado AS (
    SELECT
        nombre_evento,
        usuarios_unicos,
        LAG(usuarios_unicos) OVER (
            ORDER BY CASE nombre_evento
                WHEN 'first_visit' THEN 1
                WHEN 'add_to_cart' THEN 2
                WHEN 'select_item' THEN 3
                WHEN 'begin_checkout' THEN 4
                WHEN 'add_payment_info' THEN 5
                WHEN 'purchase' THEN 6
            END
        ) AS usuarios_anterior
    FROM funnel
)
SELECT
    nombre_evento,
    usuarios_unicos,
    usuarios_anterior,
    ROUND(usuarios_unicos * 100.0 / usuarios_anterior, 2) AS conversion_pct
FROM funnel_ordenado
ORDER BY CASE nombre_evento
    WHEN 'first_visit' THEN 1
    WHEN 'add_to_cart' THEN 2
    WHEN 'select_item' THEN 3
    WHEN 'begin_checkout' THEN 4
    WHEN 'add_payment_info' THEN 5
    WHEN 'purchase' THEN 6
END;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,nombre_evento,usuarios_unicos,usuarios_anterior,conversion_pct
0,first_visit,7796,NaN,NaN
1,add_to_cart,7634,7796.0,97.92
2,select_item,7582,7634.0,99.32
3,begin_checkout,7208,7582.0,95.07
4,add_payment_info,6250,7208.0,86.71
5,purchase,6240,6250.0,99.84


# Conclusión del funnel: 
La mayor caída de usuarios ocurre entre begin_checkout y add_payment_info, donde la tasa de conversión es de 86.71%. En esta transición pasan de 7,208 a 6,250 usuarios, lo que representa una pérdida de 958 usuarios y constituye el principal punto de abandono identificado en el funnel.

La conversión más alta corresponde a la transición entre add_payment_info y purchase, con 99.84%. Esto indica que, de los usuarios que llegan a proporcionar su información de pago, la gran mayoría completa posteriormente la compra.

Finalmente, la conversión general del funnel, desde first_visit hasta purchase, es de aproximadamente 80.04%, ya que 6,240 de los 7,796 usuarios registrados en la primera etapa terminan realizando una compra.

Los resultados muestran que la principal oportunidad de mejora se encuentra en la transición de begin_checkout a add_payment_info. Los datos permiten identificar este punto de abandono, aunque sería necesario analizar información adicional para determinar las causas específicas de esta pérdida.

---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [70]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [71]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT * 
FROM user_activity;
'''

user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [72]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH cohortes AS (
    SELECT
        id_usuario,
        DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS cohorte
    FROM users
),
actividad AS (
    SELECT
        id_usuario,
        dias_despues_registro,
        activo
    FROM user_activity
)
SELECT
    c.cohorte,
    a.dias_despues_registro / 7 AS semana,
    COUNT(DISTINCT CASE WHEN a.activo = 1 THEN a.id_usuario END) AS usuarios_activos
FROM cohortes c
JOIN actividad a
    ON c.id_usuario = a.id_usuario
GROUP BY
    c.cohorte,
    a.dias_despues_registro / 7
ORDER BY
    c.cohorte,
    semana;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte,semana,usuarios_activos
0,2025-01-01 00:00:00+00:00,1,697
1,2025-01-01 00:00:00+00:00,2,668
2,2025-01-01 00:00:00+00:00,3,656
3,2025-01-01 00:00:00+00:00,4,671
4,2025-02-01 00:00:00+00:00,1,611
5,2025-02-01 00:00:00+00:00,2,609
6,2025-02-01 00:00:00+00:00,3,635
7,2025-02-01 00:00:00+00:00,4,575
8,2025-03-01 00:00:00+00:00,1,677
9,2025-03-01 00:00:00+00:00,2,705


In [73]:
# Tamaño inicial de cada cohorte
# ==============================

query_cohort_size = '''
SELECT
    DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS cohorte,
    COUNT(DISTINCT id_usuario) AS usuarios_iniciales
FROM users
GROUP BY 
    DATE_TRUNC('month', CAST(fecha_registro AS DATE))
ORDER BY cohorte;
'''

# Ejecutar la consulta
cohorte_size = pd.read_sql(query_cohort_size, con=engine)
cohorte_size

,cohorte,usuarios_iniciales
0,2025-01-01 00:00:00+00:00,1627
1,2025-02-01 00:00:00+00:00,1444
2,2025-03-01 00:00:00+00:00,1636
3,2025-04-01 00:00:00+00:00,1606
4,2025-05-01 00:00:00+00:00,1687


In [74]:
# Porcentaje de retención por cohortes
# =====================================

query_cohort_retention_pct = '''
WITH cohortes AS (
    SELECT
        id_usuario,
        DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS cohorte
    FROM users
),
actividad AS (
    SELECT
        id_usuario,
        dias_despues_registro,
        activo
    FROM user_activity
),
usuarios_activos AS (
    SELECT
        c.cohorte,
        a.dias_despues_registro / 7 AS semana,
        COUNT(DISTINCT CASE WHEN a.activo = 1 THEN a.id_usuario END) AS usuarios_activos
    FROM cohortes c
    JOIN actividad a
        ON c.id_usuario = a.id_usuario
    GROUP BY
        c.cohorte,
        a.dias_despues_registro / 7
),
tamanio_cohorte AS (
    SELECT
        DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS cohorte,
        COUNT(DISTINCT id_usuario) AS usuarios_iniciales
    FROM users
    GROUP BY
        DATE_TRUNC('month', CAST(fecha_registro AS DATE))
)
SELECT
    ua.cohorte,
    ua.semana,
    ua.usuarios_activos,
    tc.usuarios_iniciales,
    ROUND(
        ua.usuarios_activos * 100.0 / tc.usuarios_iniciales,
        2
    ) AS porcentaje_retencion
FROM usuarios_activos ua
JOIN tamanio_cohorte tc
    ON ua.cohorte = tc.cohorte
WHERE ua.semana IN (1, 2, 3)
ORDER BY
    ua.cohorte,
    ua.semana;
'''

# Ejecutar la consulta
cohorte_retention_pct = pd.read_sql(query_cohort_retention_pct, con=engine)
cohorte_retention_pct

,cohorte,semana,usuarios_activos,usuarios_iniciales,porcentaje_retencion
0,2025-01-01 00:00:00+00:00,1,697,1627,42.84
1,2025-01-01 00:00:00+00:00,2,668,1627,41.06
2,2025-01-01 00:00:00+00:00,3,656,1627,40.32
3,2025-02-01 00:00:00+00:00,1,611,1444,42.31
4,2025-02-01 00:00:00+00:00,2,609,1444,42.17
5,2025-02-01 00:00:00+00:00,3,635,1444,43.98
6,2025-03-01 00:00:00+00:00,1,677,1636,41.38
7,2025-03-01 00:00:00+00:00,2,705,1636,43.09
8,2025-03-01 00:00:00+00:00,3,690,1636,42.18
9,2025-04-01 00:00:00+00:00,1,680,1606,42.34


# Conclusión del análisis de cohortes

La retención de usuarios se mantiene relativamente estable entre 40% y 43% durante las primeras semanas, sin presentar una caída pronunciada. Esto significa que aproximadamente 4 de cada 10 usuarios continúan activos y regresan a la plataforma, mientras que una parte importante deja de estar activa.

En cuanto al tamaño de las cohortes, enero, marzo, abril y mayo presentan cantidades similares de usuarios iniciales, entre 1,606 y 1,687, mientras que febrero registra 1,444 usuarios, una cantidad menor que el resto. Esto podría indicar una menor captación de usuarios durante ese mes, por lo que sería conveniente investigar qué factores pudieron influir.

Desde el punto de vista del negocio, la retención estable es una señal positiva, ya que existe un grupo de usuarios que continúa utilizando la plataforma. Sin embargo, el hecho de que solo alrededor del 40%-43% permanezca activo también representa una oportunidad de mejora. RappiPlus podría implementar estrategias para recuperar usuarios inactivos y fomentar su regreso, con el objetivo de aumentar la retención y fortalecer la fidelización de los usuarios.


---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):**
    No existe una diferencia estadísticamente significativa en la tasa de conversión entre los grupos control y tratamiento.
     
   - **H₁ (Hipótesis alternativa):**
    Existe una diferencia estadísticamente significativa en la tasa de conversión entre los grupos control y tratamiento.
   
**Test estadístico:**
    Test Z de dos proporciones (two-proportion z-test).

**Nivel de significancia alpha:** 
    0.05.

In [75]:
# tu código aquí
experiment = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')

display(experiment.head())
experiment.info()

,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_usuario       10000 non-null  object 
 1   variante         10000 non-null  object 
 2   convirtio        10000 non-null  int64  
 3   dispositivo      10000 non-null  object 
 4   pais             10000 non-null  object 
 5   duracion_sesion  10000 non-null  float64
 6   timestamp        10000 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 547.0+ KB


In [76]:
experiment['variante'].value_counts()

tratamiento    5035
control        4965
Name: variante, dtype: int64

In [77]:
experiment.groupby('variante')['convirtio'].mean()

variante
control        0.156898
tratamiento    0.162860
Name: convirtio, dtype: float64

In [78]:
from statsmodels.stats.proportion import proportions_ztest

# Conversiones por grupo
conversiones = experiment.groupby('variante')['convirtio'].sum()

# Número de usuarios por grupo
usuarios = experiment['variante'].value_counts()

# Ordenamos ambos para que coincidan: control y tratamiento
conversiones = conversiones[['control', 'tratamiento']]
usuarios = usuarios[['control', 'tratamiento']]

# Test de proporciones
stat, p_value = proportions_ztest(
    conversiones,
    usuarios
)

print('Conversiones:')
print(conversiones)

print('\nUsuarios:')
print(usuarios)

print('\nEstadístico Z:', stat)
print('P-value:', p_value)

Conversiones:
variante
control        779
tratamiento    820
Name: convirtio, dtype: int64

Usuarios:
control        4965
tratamiento    5035
Name: variante, dtype: int64

Estadístico Z: -0.8132782986429474
P-value: 0.41605851639119995


# Conclusión del experimento A/B

El grupo de tratamiento presentó una tasa de conversión ligeramente superior a la del grupo de control: 16.29% frente a 15.69%, lo que representa una diferencia de +0.60 puntos porcentuales.

Sin embargo, el test estadístico obtuvo un p-value de 0.4161, superior al nivel de significancia establecido. Por lo tanto, no se rechaza la hipótesis nula y no existe evidencia estadísticamente significativa para afirmar que el cambio de UI haya mejorado la tasa de conversión.

Por esta razón, no se recomienda implementar el nuevo diseño basándose únicamente en estos resultados. La diferencia observada podría deberse a la variación aleatoria. No obstante, tampoco sería necesario descartar inmediatamente la propuesta: podría ser conveniente analizar el comportamiento por segmentos, como dispositivo o país, y realizar pruebas adicionales antes de tomar una decisión definitiva.

---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard

In [166]:
# (Pega aquí tu link)
# link de power bi o tableau
# link de one drive / google drive

https://drive.google.com/drive/folders/11-fYN6Ls45sA8Rtoh3rl8iJZP0H8XCiD?usp=sharing